1. Import file to clean

In [106]:
from google.colab import files
import pandas as pd
import re

uploaded = files.upload()

for filename in uploaded.keys():
    print("Uploaded:", filename)

Saving data_to_clean_tours.csv to data_to_clean_tours (1).csv
Uploaded: data_to_clean_tours (1).csv


2. Initial Inspection

In [107]:
df = pd.read_csv(filename)

df.head()


,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],NaN,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]


In [108]:
df.info()
df.isnull().sum()
df.duplicated().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Rank                              20 non-null     int64 
 1   Peak                              9 non-null      object
 2   All Time Peak                     6 non-null      object
 3   Actual gross                      20 non-null     object
 4   Adjusted gross (in 2022 dollars)  20 non-null     object
 5   Artist                            20 non-null     object
 6   Tour title                        20 non-null     object
 7   Year(s)                           20 non-null     object
 8   Shows                             20 non-null     int64 
 9   Average gross                     20 non-null     object
 10  Ref.                              20 non-null     object
dtypes: int64(2), object(9)
memory usage: 1.8+ KB


np.int64(0)

In [109]:
df.columns = (
    df.columns
      .str.replace('\u00A0', ' ', regex=False)
      .str.strip()
)

3. Standardize Column Names

In [110]:
df = df.rename(columns={
    'Actual gross': 'Actual_Gross',
    'Adjusted gross (in 2022 dollars)': 'Adjusted_Gross',
    'Average gross': 'Average_Gross',
    'Tour title': 'Tour_Title',
    'All Time Peak': 'All_Time_Peak',
})

4. Identify Data Quality Issues

In [114]:
df['Peak'] = df['Peak'].astype(str).str.replace(r'\[.*?\]', '', regex=True)
df['All_Time_Peak'] = df['All_Time_Peak'].astype(str).str.replace(r'\[.*?\]', '', regex=True)

df['Peak'] = pd.to_numeric(df['Peak'], errors='coerce')
df['All_Time_Peak'] = pd.to_numeric(df['All_Time_Peak'], errors='coerce')

In [115]:
df[['Peak', 'All_Time_Peak']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Peak           9 non-null      float64
 1   All_Time_Peak  6 non-null      float64
dtypes: float64(2)
memory usage: 452.0 bytes


5. Clean Citation Artifacts

    5a. Clean Currency Fields

In [116]:
# Function to clean and convert currency columns to numeric
def clean_currency(column):
    return column.astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()

# Apply cleaning and conversion to monetary columns, using the new column names
for col in ['Actual_Gross', 'Adjusted_Gross', 'Average_Gross']:
    df[col] = clean_currency(df[col])
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Display the first few rows of the cleaned DataFrame and check info to confirm changes
display(df.head())
df.info()

,Rank,Peak,All_Time_Peak,Actual_Gross,Adjusted_Gross,Artist,Tour_Title,Year(s),Shows,Average_Gross,Ref.
0,1,1.0,2.0,780000000.0,780000000,Taylor Swift,The Eras Tour †,2023–2024,56,13928571,[1]
1,2,1.0,7.0,579800000.0,579800000,Beyoncé,Renaissance World Tour,2023,56,10353571,[3]
2,3,1.0,2.0,411000000.0,560622615,Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,4835294,[6]
3,4,2.0,10.0,397300000.0,454751555,Pink,Beautiful Trauma World Tour,2018–2019,156,2546795,[7]
4,5,2.0,NaN,345675146.0,402844849,Taylor Swift,Reputation Stadium Tour,2018,53,6522173,[8]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Rank            20 non-null     int64  
 1   Peak            9 non-null      float64
 2   All_Time_Peak   6 non-null      float64
 3   Actual_Gross    18 non-null     float64
 4   Adjusted_Gross  20 non-null     int64  
 5   Artist          20 non-null     object 
 6   Tour_Title      20 non-null     object 
 7   Year(s)         20 non-null     object 
 8   Shows           20 non-null     int64  
 9   Average_Gross   20 non-null     int64  
 10  Ref.            20 non-null     object 
dtypes: float64(3), int64(4), object(4)
memory usage: 1.8+ KB


    5b. Remove Wikipedia Artifacts

In [117]:
df['Tour_Title'] = df['Tour_Title'].str.replace(
    r'[†‡]',
    '',
    regex=True
)

In [118]:
df['Tour_Title'] = df['Tour_Title'].str.strip()

In [119]:
for col in df.select_dtypes(include='object'):
    df[col] = (
        df[col]
        .str.replace(r'\[.*?\]', '', regex=True)  # remove [1], [a], etc.
        .str.replace(r'\*', '', regex=True)       # remove *
        .str.strip()                              # remove extra spaces
    )
  # Note: didn't remove () because it might be a song/tour (Ex. (Don’t Fear) The Reaper)

In [120]:
for col in df.select_dtypes(include='object'):
    mask = df[col].astype(str).str.contains(r'\[.*?\]|\*', regex=True, na=False)

    if mask.any():
        print(f"Artifacts remain in {col}")

In [121]:
print(df['Peak'].unique())
print(df['All_Time_Peak'].unique())

[ 1.  2. nan]
[ 2.  7. 10. nan 14.]


6. Handle Missing Values

Missing ranking information was preserved as NaN because no reliable replacement values were available.

The Wikipedia reference column was removed because it is not relevant for analysis. If analyzing only actual gross revenue, rows with missing Actual_Gross values could be removed using dropna().

In [122]:
df.drop(columns='Ref.', inplace=True)
# df_gross = df_clean.dropna(subset=['Actual gross'])

7. Feature Engineering

In [123]:
# Splitting Years
df[['Start Year', 'End Year']] = (
    df['Year(s)']
    .str.split('–', expand=True)
)
df['End Year'] = df['End Year'].fillna(df['Start Year'])

df['Start Year'] = pd.to_numeric(df['Start Year'])
df['End Year'] = pd.to_numeric(df['End Year'])
df.drop(columns='Year(s)', inplace=True)

8. Data Validation

Verify that no unexpected missing values remain and confirm that Average_Gross is consistent with Actual_Gross / Shows.

In [124]:
df.isna().sum()

,0
Rank,0
Peak,11
All_Time_Peak,14
Actual_Gross,2
Adjusted_Gross,0
Artist,0
Tour_Title,0
Shows,0
Average_Gross,0
Start Year,0


In [125]:
df.describe()

,Rank,Peak,All_Time_Peak,Actual_Gross,Adjusted_Gross,Shows,Average_Gross,Start Year,End Year
count,20.000000,9.000000,6.000000,1.800000e+01,2.000000e+01,20.000000,2.000000e+01,20.00000,20.000000
mean,10.450000,1.555556,7.500000,2.979010e+08,3.438781e+08,110.000000,3.726571e+06,2013.85000,2014.700000
std,5.942488,0.527046,4.806246,1.617236e+08,1.514627e+08,66.507617,3.393340e+06,5.62209,5.332127
min,1.000000,1.000000,2.000000,1.500000e+08,1.854231e+08,41.000000,6.153850e+05,2002.00000,2005.000000
25%,5.750000,1.000000,3.250000,1.955000e+08,2.457557e+08,59.000000,1.647508e+06,2011.25000,2011.750000
50%,10.500000,2.000000,8.500000,2.532423e+08,2.974889e+08,87.000000,2.342100e+06,2013.50000,2014.500000
75%,15.250000,2.000000,10.000000,3.355460e+08,3.924451e+08,134.500000,4.933024e+06,2016.50000,2017.250000
max,20.000000,2.000000,14.000000,7.800000e+08,7.800000e+08,325.000000,1.392857e+07,2023.00000,2024.000000


In [126]:
df['Calculated_Avg'] = df['Actual_Gross'] / df['Shows']

In [127]:
df[['Average_Gross', 'Calculated_Avg']].head()

,Average_Gross,Calculated_Avg
0,13928571,1.392857e+07
1,10353571,1.035357e+07
2,4835294,4.835294e+06
3,2546795,2.546795e+06
4,6522173,6.522173e+06


In [128]:
df['Peak'] = df['Peak'].astype('Int64')
df['All_Time_Peak'] = df['All_Time_Peak'].astype('Int64')

9. Final Validation

In [129]:
print(df.shape)

df.info()

df.isna().sum()

(20, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Rank            20 non-null     int64  
 1   Peak            9 non-null      Int64  
 2   All_Time_Peak   6 non-null      Int64  
 3   Actual_Gross    18 non-null     float64
 4   Adjusted_Gross  20 non-null     int64  
 5   Artist          20 non-null     object 
 6   Tour_Title      20 non-null     object 
 7   Shows           20 non-null     int64  
 8   Average_Gross   20 non-null     int64  
 9   Start Year      20 non-null     int64  
 10  End Year        20 non-null     int64  
 11  Calculated_Avg  18 non-null     float64
dtypes: Int64(2), float64(2), int64(6), object(2)
memory usage: 2.0+ KB


,0
Rank,0
Peak,11
All_Time_Peak,14
Actual_Gross,2
Adjusted_Gross,0
Artist,0
Tour_Title,0
Shows,0
Average_Gross,0
Start Year,0


10. Export Cleaned Dataset

In [130]:
df.to_csv('cleaned_tours.csv', index=False)
files.download('cleaned_tours.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>